# Machine Learning for Classification

We'll use logistic regression to predict churn

# 3.1 Churn prediction project
Dataset: https://www.kaggle.com/blastchar/telco-customer-churn

https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv


In [104]:
import pandas as pd
import numpy as np
import matplotlib as plt

In [105]:
data = "https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv"

In [106]:
!wget $data -O churn-data.csv

--2026-05-22 15:21:00--  https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8003::154, 2606:50c0:8000::154, 2606:50c0:8001::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8003::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 977501 (955K) [text/plain]
Saving to: ‘churn-data.csv’

churn-data.csv      100%[===================>] 954.59K  --.-KB/s    in 0.07s   

2026-05-22 15:21:00 (14.2 MB/s) - ‘churn-data.csv’ saved [977501/977501]



In [107]:
df = pd.read_csv('churn-data.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [108]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
categorical_columns = list(df.dtypes[df.dtypes == 'str'].index)
for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')


In [109]:
df.dtypes

customerid              str
gender                  str
seniorcitizen         int64
partner                 str
dependents              str
tenure                int64
phoneservice            str
multiplelines           str
internetservice         str
onlinesecurity          str
onlinebackup            str
deviceprotection        str
techsupport             str
streamingtv             str
streamingmovies         str
contract                str
paperlessbilling        str
paymentmethod           str
monthlycharges      float64
totalcharges            str
churn                   str
dtype: object

In [110]:
df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges = df.totalcharges.fillna(0)

In [111]:
df.churn.head()
df.churn = (df.churn == "yes" ).astype(int)

In [112]:
df.churn.head()

0    0
1    0
2    1
3    0
4    1
Name: churn, dtype: int64

# 3.3 Setting up the validation framework
* Perform the train/validation/test split with Scikit-Learn

In [113]:
from sklearn.model_selection import train_test_split

In [114]:
df_full_train, df_test = train_test_split(df, test_size = 0.2, random_state = 1)


In [115]:
df_train, df_val = train_test_split(df_full_train, test_size = 0.25, random_state = 1)

In [116]:
len(df_train), len(df_val), len(df_test)

(4225, 1409, 1409)

In [117]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [118]:
y_train = df_train.churn.values
y_val = df_val.churn.values
y_test = df_test.churn.values

In [119]:
y_train, y_val, y_test

(array([0, 0, 1, ..., 1, 0, 1], shape=(4225,)),
 array([0, 0, 0, ..., 0, 1, 1], shape=(1409,)),
 array([0, 0, 0, ..., 0, 0, 1], shape=(1409,)))

In [120]:
del df_train['churn']
del df_test['churn']
del df_val['churn']

In [121]:
df_full_train.churn.value_counts(normalize=True)

churn
0    0.730032
1    0.269968
Name: proportion, dtype: float64

* **The churn rate is 27%**

In [122]:
global_churn_rate = df_full_train.churn.mean()
global_churn_rate

np.float64(0.26996805111821087)

In [123]:
df_full_train.dtypes

customerid              str
gender                  str
seniorcitizen         int64
partner                 str
dependents              str
tenure                int64
phoneservice            str
multiplelines           str
internetservice         str
onlinesecurity          str
onlinebackup            str
deviceprotection        str
techsupport             str
streamingtv             str
streamingmovies         str
contract                str
paperlessbilling        str
paymentmethod           str
monthlycharges      float64
totalcharges        float64
churn                 int64
dtype: object

In [124]:
numerical = ['tenure', 'monthlycharges', 'totalcharges']

In [128]:
categorical = [
    'gender',
    'seniorcitizen',
    'partner',
    'dependents',
    'phoneservice',
    'multiplelines',
    'internetservice',
    'onlinesecurity',
    'onlinebackup',
    'deviceprotection',
    'techsupport',
    'streamingtv',
    'streamingmovies',
    'contract',
    'paperlessbilling',
    'paymentmethod',
]

In [131]:
# look at the unique values in our categorical subset 
df_full_train[categorical].nunique()

gender              2
seniorcitizen       2
partner             2
dependents          2
phoneservice        2
multiplelines       3
internetservice     3
onlinesecurity      3
onlinebackup        3
deviceprotection    3
techsupport         3
streamingtv         3
streamingmovies     3
contract            3
paperlessbilling    2
paymentmethod       4
dtype: int64

# 3.5 Feature Importance

## What is Feature Importance?

Feature importance means checking which features are most related to the target variable.

In this case, the target is churn.

## Churn Rate

Churn rate is the percentage of customers who leave.

Example:

- 100 customers have monthly contracts
- 42 customers churned

Churn rate = 42%

## Risk Ratio

Risk ratio compares the churn rate of one group to the average churn rate.

Example:

- overall churn rate = 25%
- monthly contract churn rate = 42%

Risk ratio = 42 / 25 = 1.68

This means monthly contract customers are 1.68 times more likely to churn than average.
$$
\text{Risk Ratio} = \frac{\text{Churn Rate of Specific Segment}}{\text{Global Churn Rate}}
$$
## Interpreting Risk Ratio
- If risk ratio = 1.0, the segment churns at the same rate as the average customer.
- If risk ratio > 1.0, the segment is more likely to churn than the average customer.

- If risk ratio < 1.0, the segment is less likely to churn than the average customer.



## Summary

Feature importance helps us find which customer details affect churn.

Churn rate shows how many customers leave in each group.

Risk ratio compares each group to the average customer.

Mutual information gives a more mathematical score for how useful each feature is.

In [135]:
churn_female = df_full_train[df_full_train.gender == 'female'].churn.mean()
churn

np.float64(0.27682403433476394)

In [136]:
churn_male = df_full_train[df_full_train.gender == 'male'].churn.mean()
churn_male

np.float64(0.2632135306553911)

In [137]:
global_churn = df_full_train.churn.mean()
global_churn

np.float64(0.26996805111821087)

In [152]:
global_churn / churn_male

np.float64(1.0256617524410743)

In [153]:
global_churn / churn_female 

np.float64(0.975233424969661)

**we see that the gender does not have an affect**

In [142]:
churn_partner = df_full_train[df_full_train.partner == 'yes'].churn.mean()
churn_partner

np.float64(0.20503330866025166)

In [147]:
chur_no_partner = df_full_train[df_full_train.partner == 'no'].churn.mean()
chur_no_partner

np.float64(0.3298090040927694)

In [151]:
global_churn / chur_no_partner

np.float64(0.8185587651278121)

In [150]:
global_churn / churn_partner

np.float64(1.3167033828906243)

In [157]:
df_group = df_full_train.groupby('gender').churn.agg(['mean', 'count'])
df_group['diff'] = df_group['mean'] - global_churn
df_group['risk'] = df_group['mean'] / global_churn
df_group

,mean,count,diff,risk
gender,,,,
female,0.276824,2796,0.006856,1.025396
male,0.263214,2838,-0.006755,0.974980


In [159]:
categorical

['gender',
 'seniorcitizen',
 'partner',
 'dependents',
 'phoneservice',
 'multiplelines',
 'internetservice',
 'onlinesecurity',
 'onlinebackup',
 'deviceprotection',
 'techsupport',
 'streamingtv',
 'streamingmovies',
 'contract',
 'paperlessbilling',
 'paymentmethod']

In [161]:
from IPython.display import display 

In [168]:
for c in categorical:
    df_group = df_full_train.groupby(c).churn.agg(['mean', 'count'])
    df_group['diff'] = df_group['mean'] - global_churn
    df_group['risk'] = df_group['mean'] / global_churn
    display(df_group)

,mean,count,diff,risk
gender,,,,
female,0.276824,2796,0.006856,1.025396
male,0.263214,2838,-0.006755,0.974980


,mean,count,diff,risk
seniorcitizen,,,,
0,0.242270,4722,-0.027698,0.897403
1,0.413377,912,0.143409,1.531208


,mean,count,diff,risk
partner,,,,
no,0.329809,2932,0.059841,1.221659
yes,0.205033,2702,-0.064935,0.759472


,mean,count,diff,risk
dependents,,,,
no,0.313760,3968,0.043792,1.162212
yes,0.165666,1666,-0.104302,0.613651


,mean,count,diff,risk
phoneservice,,,,
no,0.241316,547,-0.028652,0.893870
yes,0.273049,5087,0.003081,1.011412


...

,mean,count,diff,risk
streamingtv,,,,
no,0.342832,2246,0.072864,1.269897
no_internet_service,0.077805,1221,-0.192163,0.288201
yes,0.302723,2167,0.032755,1.121328


,mean,count,diff,risk
streamingmovies,,,,
no,0.338906,2213,0.068938,1.255358
no_internet_service,0.077805,1221,-0.192163,0.288201
yes,0.307273,2200,0.037305,1.138182


,mean,count,diff,risk
contract,,,,
month-to-month,0.431701,3104,0.161733,1.599082
one_year,0.120573,1186,-0.149395,0.446621
two_year,0.028274,1344,-0.241694,0.104730


,mean,count,diff,risk
paperlessbilling,,,,
no,0.172071,2313,-0.097897,0.637375
yes,0.338151,3321,0.068183,1.252560


,mean,count,diff,risk
paymentmethod,,,,
bank_transfer_(automatic),0.168171,1219,-0.101797,0.622928
credit_card_(automatic),0.164339,1217,-0.105630,0.608733
electronic_check,0.455890,1893,0.185922,1.688682
mailed_check,0.193870,1305,-0.076098,0.718121


## 3.6 Mutual Information

Mutual information is another way to measure feature importance.

It tells us how useful a feature is for predicting churn.

Higher mutual information means the feature gives more information about churn.

In [169]:
from sklearn.metrics import mutual_info_score

In [170]:
mutual_info_score(df_full_train.partner, df_full_train.churn)

0.009967689095399745

In [172]:
def mutual_info_churn_score(series):
    return mutual_info_score(series, df_full_train.churn)

In [175]:
mi = df_full_train[categorical].apply(mutual_info_churn_score)
mi.sort_values(ascending=False)

contract            0.098320
onlinesecurity      0.063085
techsupport         0.061032
internetservice     0.055868
onlinebackup        0.046923
deviceprotection    0.043453
paymentmethod       0.043210
streamingtv         0.031853
streamingmovies     0.031581
paperlessbilling    0.017589
dependents          0.012346
partner             0.009968
seniorcitizen       0.009410
multiplelines       0.000857
phoneservice        0.000229
gender              0.000117
dtype: float64

# 3.7 Feature importance: Correlation

Correlation is a way to measure how strongly a numerical feature is related to the target variable.

In the churn problem, the target is `churn`.

Usually, churn is converted into numbers:

- No churn = 0
- Churn = 1

Correlation gives a value between -1 and +1:

$$
-1 \leq r \leq 1
$$

## Positive Correlation

If:

$$
r > 0
$$

then as the feature increases, churn tends to increase.

Example:

- `monthlycharges` has positive correlation with churn.
- Customers with higher monthly charges may be more likely to churn.

## Negative Correlation

If:

$$
r < 0
$$

then as the feature increases, churn tends to decrease.

Example:

- `tenure` has negative correlation with churn.
- Customers who stayed longer are less likely to churn.

## Weak Correlation

If:

$$
r \approx 0
$$

then there is no strong linear relationship between the feature and churn.

## Why We Use Correlation

Correlation helps us find which numerical features are related to the target.

It is useful for:

- understanding the data
- finding important numerical features
- detecting patterns
- explaining model behavior
- deciding which features may be useful

## Correlation vs Mutual Information

Mutual information is usually used for categorical features.

Examples:

- contract
- internetservice
- paymentmethod

Correlation is usually used for numerical features.

Examples:

- tenure
- monthlycharges
- totalcharges

## Important Notes

Correlation measures relationship, not causation.

A strong correlation does not prove that one variable directly causes another.

Correlation is also mainly useful for linear relationships. A feature can still be useful even if its correlation is close to zero.

## Summary

Feature importance helps us understand which features are most related to the target.

For categorical features, we can use:

- churn rate
- risk ratio
- mutual information

For numerical features, we can use:

- correlation

Correlation tells us whether a numerical feature increases or decreases together with churn.

In [176]:
df_full_train[numerical].corrwith(df_full_train.churn)

tenure           -0.351885
monthlycharges    0.196805
totalcharges     -0.196353
dtype: float64

In this case, the target is:

- `churn = 1` → customer churned
- `churn = 0` → customer did not churn

The correlation values are:

```text
tenure           -0.351885: tenure increase → churn decrease
monthlycharges    0.196805: monthly charges increase → churn increases
totalcharges     -0.196353: totalcharges increase → churn decrease


In [179]:
df_full_train[df_full_train.tenure <= 2].churn.mean()

np.float64(0.5953420669577875)

In [180]:
df_full_train[(df_full_train.tenure > 2) & (df_full_train.tenure <= 12)].churn.mean()

np.float64(0.3994413407821229)

In [181]:
df_full_train[df_full_train.tenure > 12].churn.mean()

np.float64(0.17634908339788277)

In [182]:
df_full_train[(df_full_train.monthlycharges > 20) & (df_full_train.monthlycharges <= 50)].churn.mean()

np.float64(0.18340943683409436)

# 3.8 One-Hot Encoding

One-hot encoding is a way to convert categorical text values into numerical columns.

Machine learning models need numbers, so categories like:

- month-to-month
- one year
- two year

are converted into separate 0/1 columns.

Example:

| contract | contract_month-to-month | contract_one year | contract_two year |
|---|---:|---:|---:|
| month-to-month | 1 | 0 | 0 |
| one year | 0 | 1 | 0 |
| two year | 0 | 0 | 1 |

A value of `1` means the category is present.

A value of `0` means the category is not present.

We use one-hot encoding because assigning numbers like 1, 2, and 3 to categories can mislead the model into thinking the categories have an order.

One-hot encoding is commonly used for categorical features such as:

- contract
- internetservice
- paymentmethod
- gender

In churn prediction, it helps the model learn patterns such as:

- month-to-month contracts may have higher churn risk
- two-year contracts may have lower churn risk

One-hot encoding is usually done before training the model.

In [183]:
from sklearn.feature_extraction import DictVectorizer

In [187]:
df_train[['gender','contract']].iloc[:10].to_dict(orient='records')

[{'gender': 'female', 'contract': 'two_year'},
 {'gender': 'male', 'contract': 'month-to-month'},
 {'gender': 'female', 'contract': 'month-to-month'},
 {'gender': 'female', 'contract': 'month-to-month'},
 {'gender': 'female', 'contract': 'two_year'},
 {'gender': 'male', 'contract': 'month-to-month'},
 {'gender': 'male', 'contract': 'month-to-month'},
 {'gender': 'female', 'contract': 'month-to-month'},
 {'gender': 'female', 'contract': 'two_year'},
 {'gender': 'female', 'contract': 'month-to-month'}]

In [199]:
dicts = df_train[['gender','contract']].iloc[:10].to_dict(orient = 'records')

In [200]:
dv = DictVectorizer(sparse=False)

In [201]:
dv.fit(dicts)

,"dtype dtype: dtype, default=np.float64The type of feature values. Passed to Numpy array/scipy.sparse matrixconstructors as the dtype argument.",<class 'numpy.float64'>
,"separator separator: str, default=""=""Separator string used when constructing new features for one-hotcoding.",'='
,"sparse sparse: bool, default=TrueWhether transform should produce scipy.sparse matrices.",False
,"sort sort: bool, default=TrueWhether ``feature_names_`` and ``vocabulary_`` should besorted when fitting.",True


In [202]:
dv.transform(dicts)

array([[0., 1., 1., 0.],
       [1., 0., 0., 1.],
       [1., 0., 1., 0.],
       [1., 0., 1., 0.],
       [0., 1., 1., 0.],
       [1., 0., 0., 1.],
       [1., 0., 0., 1.],
       [1., 0., 1., 0.],
       [0., 1., 1., 0.],
       [1., 0., 1., 0.]])